# **Pengantar Data Sains | 21 Februari 2026**

DATA STRUCTURES & TYPES: Variables & Observations; Measurement Scales; Cross-Sectional, Time-Series, dan Panel Data

**Learning outcomes**
- Membedakan variable vs observation dan bentuk data (wide/long).
- Mengidentifikasi skala pengukuran (nominal, ordinal, interval, ratio) dan implikasi analisisnya.
- Membedakan cross-sectional, time-series, dan panel data, serta operasi dasar tiap tipe (filtering, agregasi, reshaping).
- Mengimplementasikan struktur data dan tipe data di R dan Python secara konsisten.

## 0) Setup

R

`# install.packages(c("tidyverse", "lubridate", "slider"))`

`library(tidyverse)`

`library(lubridate)`

`library(slider)`

`set.seed(123)`

In [ ]:
import numpy as np
import pandas as pd
from datetime import date, datetime, timedelta
np.random.seed(123)

## 1) Dataset mini

Dataset mencakup:
- Cross-sectional: individu (id) + karakteristik (gender, pendidikan/ordinal) + outcome (income/ratio).
- Time-series: date + variabel time-series (price/ratio).
- Panel: (id, date) + outcome (spend/ratio).

In [ ]:
import numpy as np
import pandas as pd
np.random.seed(123)

# --- cross-sectional ---
df_cross = pd.DataFrame({
"id": np.arange(1, 9),
"gender": np.random.choice(["F", "M"], size=8, replace=True), # nominal
"edu": np.random.choice(["SMA", "S1", "S2"], size=8, replace=True),# ordinal
"age": np.random.randint(20, 46, size=8), # ratio
"income": np.round(np.random.lognormal(mean=np.log(5), sigma=0.3, size=8), 2)
})
# --- time-series ---
dates = pd.date_range("2025-01-01", periods=10, freq="D")
price = np.round(np.cumsum(np.random.normal(loc=0.2, scale=1, size=10)) + 100, 2)
df_ts = pd.DataFrame({"date": dates, "price": price})
# --- panel ---
df_panel = (
pd.MultiIndex.from_product([range(1, 5), pd.date_range("2025-01-01",
periods=5, freq="D")],
names=["id", "date"])
.to_frame(index=False)
)
df_panel["spend"] = np.round(np.random.gamma(shape=5, scale=1/0.8,
size=len(df_panel)), 2)

In [ ]:
df_cross

,id,gender,edu,age,income
0,1,F,S2,20,7.82
1,2,M,S1,34,4.13
2,3,F,S2,20,4.38
3,4,F,S1,35,4.39
4,5,F,SMA,45,9.69
5,6,F,S1,39,9.64
6,7,F,S2,34,6.76
7,8,M,S1,24,5.61


In [ ]:
df_ts

,date,price
0,2025-01-01,100.94
1,2025-01-02,102.63
2,2025-01-03,101.89
3,2025-01-04,103.27
4,2025-01-05,102.21
5,2025-01-06,101.78
6,2025-01-07,102.88
7,2025-01-08,101.65
8,2025-01-09,101.71
9,2025-01-10,101.05


In [ ]:
df_panel

,id,date,spend
0,1,2025-01-01,5.17
1,1,2025-01-02,1.07
2,1,2025-01-03,8.71
3,1,2025-01-04,5.38
4,1,2025-01-05,3.77
5,2,2025-01-01,6.63
6,2,2025-01-02,4.84
7,2,2025-01-03,7.52
8,2,2025-01-04,14.97
9,2,2025-01-05,7.02


## 2) Modul Hands-On 1 — Variables & Observations (struktur data)

**Konsep inti**
- Observation = unit analisis (individu, perusahaan, hari, dsb.) -> biasanya baris (row).
- Variable = atribut yang diukur pada observation -> biasanya kolom (column).
- Wide vs Long: wide menyebar ke banyak kolom; long memakai key + value (umumnya lebih cocok untuk plotting/modeling panel/time-series).

### **Latihan 1A: cek “apa yang menjadi observation?”**

**Tugas: Jelaskan observation untuk df_cross, df_ts, df_panel; hitung jumlah observation dan jumlah variable.**

R

`dim(df_cross); names(df_cross)`

`dim(df_ts); names(df_ts)`

`dim(df_panel); names(df_panel)`

In [ ]:
df_cross.shape, df_cross.columns

((8, 5), Index(['id', 'gender', 'edu', 'age', 'income'], dtype='object'))

In [ ]:
df_ts.shape, df_ts.columns

((10, 2), Index(['date', 'price'], dtype='object'))

In [ ]:
df_panel.shape, df_panel.columns

((20, 3), Index(['id', 'date', 'spend'], dtype='object'))

### **Latihan 1B: Konversi wide ke long (panel per id)**

**Tugas: Buat data panel dalam format wide (kolom per tanggal), lalu kembalikan ke long.**

R

`panel_wide <- df_panel %>% mutate(date = as.character(date)) %>% pivot_wider(names_from = date, values_from = spend)`

`panel_long <- panel_wide %>% pivot_longer(-id, names_to = "date", values_to = "spend") %>% mutate(date = ymd(date)) %>% arrange(id, date)`

`panel_wide`

`panel_long`

In [ ]:
panel_wide = df_panel.pivot(index="id", columns="date",
values="spend").reset_index()
panel_long = (
panel_wide.melt(id_vars="id", var_name="date", value_name="spend")
.sort_values(["id","date"])
)

In [ ]:
panel_wide.head()

date,id,2025-01-01 00:00:00,2025-01-02 00:00:00,2025-01-03 00:00:00,2025-01-04 00:00:00,2025-01-05 00:00:00
0,1,5.17,1.07,8.71,5.38,3.77
1,2,6.63,4.84,7.52,14.97,7.02
2,3,8.90,14.20,11.92,3.93,8.58
3,4,11.97,3.99,8.25,10.56,8.30


In [ ]:
panel_long.head()

,id,date,spend
0,1,2025-01-01 00:00:00,5.17
4,1,2025-01-02 00:00:00,1.07
8,1,2025-01-03 00:00:00,8.71
12,1,2025-01-04 00:00:00,5.38
16,1,2025-01-05 00:00:00,3.77


## 3) Modul Hands-On 2 — Measurement Scales (nominal/ordinal/interval/ratio)

**Konsep ringkas**
- **Nominal**: Kategori tanpa urutan (mis. *gender, province*). Analisis yang sesuai meliputi mode, frekuensi/proporsi, serta uji chi-square untuk menilai asosiasi antar-variabel kategorikal.

- **Ordinal**: Kategori berurutan, tetapi jarak antar-level tidak pasti (mis. edu: SMA < S1 < S2). Analisis yang sesuai meliputi median/kuantil dan metode berbasis peringkat (rank-based), misalnya uji Mann–Whitney/Wilcoxon/Kruskal–Wallis atau korelasi Spearman.

- **Interval**: Jarak antar-nilai bermakna dan konsisten, tetapi nol tidak absolut (mis. suhu °C). Karena itu selisih (tambah/kurang) bermakna, sedangkan rasio/perbandingan (mis. “dua kali lebih besar”) tidak bermakna.

- **Ratio**: Jarak antar-nilai bermakna dan konsisten, serta memiliki nol absolut (mis. *income, spend, price*). Oleh karena itu seluruh operasi aritmetika termasuk rasio/perbandingan (kali/bagi) bermakna.

### **Latihan 2A: set tipe data yang tepat (ordinal & category)**

R

`df_cross2 <- df_cross %>%mutate(gender = factor(gender), # nominal`

`edu = factor(edu, levels = c("SMA","S1","S2"), ordered = TRUE) # ordinal)`

`str(df_cross2)`

`summary(df_cross2$edu)`

In [ ]:
df_cross2 = df_cross.copy()
df_cross2["gender"] = df_cross2["gender"].astype("category")
edu_order = pd.CategoricalDtype(categories=["SMA","S1","S2"], ordered=True)
df_cross2["edu"] = df_cross2["edu"].astype(edu_order)
df_cross2.dtypes

,0
id,int64
gender,category
edu,category
age,int64
income,float64


In [ ]:
df_cross2["edu"].cat.categories, df_cross2["edu"].cat.ordered

(Index(['SMA', 'S1', 'S2'], dtype='object'), True)

### **Latihan 2B: operasi yang “legal” per skala**

**Tugas:**
1. Hitung tabel frekuensi (dan proporsi) untuk variabel *gender*,
2. Tentukan median tingkat pendidikan dengan merepresentasikan *edu* sebagai peringkat (rank) sesuai urutan level (mis. SMA < S1 < S2),
3. Hitung rata-rata *(mean) income* dan koefisien variasi (CV = SD/mean) untuk *income*.


R

`table(df_cross2$gender)`

`median(as.integer(df_cross2$edu)) # interpretasi sebagai tingkat`

`mean(df_cross2$income)`

`sd(df_cross2$income) / mean(df_cross2$income)`

In [ ]:
df_cross2["gender"].value_counts()

,count
gender,
F,6
M,2


In [ ]:
np.median(df_cross2["edu"].cat.codes + 1) # +1 agar level mulai dari 1

np.float64(2.0)

In [ ]:
df_cross2["income"].mean()

np.float64(6.5525)

In [ ]:
df_cross2["income"].std(ddof=1) / df_cross2["income"].mean()

np.float64(0.35171725795493813)

## 4) Modul Hands-On 3 — Cross-sectional vs Time-series vs Panel

### A. Cross-sectional (satu waktu, banyak unit)

**Tugas: Hitung rata-rata income per gender**

R

`df_cross2 %>% group_by(gender) %>% summarise(mean_income = mean(income), n = n())`

In [ ]:
df_cross2.groupby("gender", observed=True).agg(
mean_income=("income","mean"),
n=("id","count")
)

,mean_income,n
gender,,
F,7.113333,6
M,4.870000,2


### B. Time-series (satu unit, banyak waktu)

**Tugas:**

1. Hitung log-return
2. Buat rolling mean 3-hari

R

`df_ts2 <- df_ts %>% arrange(date) %>% mutate(logret = log(price) - lag(log(price)), roll3 = slider::slide_dbl(price, mean, .before = 2, .complete = TRUE))`

`df_ts2`

In [ ]:
df_ts2 = df_ts.sort_values("date").copy()
df_ts2["logret"] = np.log(df_ts2["price"]).diff()
df_ts2["roll3"] = df_ts2["price"].rolling(3).mean()
df_ts2

,date,price,logret,roll3
0,2025-01-01,100.94,NaN,NaN
1,2025-01-02,102.63,0.016604,NaN
2,2025-01-03,101.89,-0.007236,101.820000
3,2025-01-04,103.27,0.013453,102.596667
4,2025-01-05,102.21,-0.010317,102.456667
5,2025-01-06,101.78,-0.004216,102.420000
6,2025-01-07,102.88,0.010750,102.290000
7,2025-01-08,101.65,-0.012028,102.103333
8,2025-01-09,101.71,0.000590,102.080000
9,2025-01-10,101.05,-0.006510,101.470000


### C. Panel data (banyak unit, banyak waktu)

**Tugas:**

1. Hitung rata-rata (mean) spend untuk setiap id;
2. Hitung pertumbuhan/perubahan harian spend untuk setiap id (mis. Δspend_t = spend_t - spend_(t-1)) ;
3. Lakukan agregasi waktu dengan menghitung rata-rata (mean) spend untuk setiap tanggal (dirata-ratakan across id).

R

`df_panel2 <- df_panel %>% arrange(id, date) %>% group_by(id) %>% mutate(d_spend = spend - lag(spend)) %>% ungroup()`

`df_panel %>% group_by(id) %>% summarise(mean_spend = mean(spend))`

`df_panel2 %>% group_by(date) %>% summarise(mean_spend = mean(spend))`

In [ ]:
df_panel2 = df_panel.sort_values(["id","date"]).copy()
df_panel2["d_spend"] = df_panel2.groupby("id")["spend"].diff()
df_panel.groupby("id").agg(mean_spend=("spend","mean"))
df_panel2.groupby("date").agg(mean_spend=("spend","mean"))

,mean_spend
date,
2025-01-01,8.1675
2025-01-02,6.0250
2025-01-03,9.1000
2025-01-04,8.7100
2025-01-05,6.9175


## 5) Lampiran Praktik — Tipe Data & Struktur Data (R dan Python)

Bagian ini memberikan pemahaman terkait dengan tipe data dan struktur data sebelum manipulasi data tabular menggunakan R dan Python.

### **5.1 Tipe data inti**

R

`x_num <- 42`

`x_int <- 10L`

`x_chr <- "42"`

`x_log <- TRUE`

`x_cpl <- 3 + 2i`

`x_fac <- factor(c("F","M","F")`)

`x_date <- Sys.Date()`

`x_time <- Sys.time()`

`typeof(x_num); typeof(x_int); typeof(x_chr); typeof(x_log); typeof(x_cpl)`

`is.factor(x_fac); levels(x_fac)`

`class(x_date); class(x_time)`

In [ ]:
x_int = 10
x_float = 3.14
x_str = "42"
x_bool = True
x_cpl = 3 + 2j
today = date.today()
now = datetime.now()
type(x_int), type(x_float), type(x_str), type(x_bool), type(x_cpl), type(today),
type(now)

datetime.datetime

### **5.2 Struktur data inti**

R

`# vector (homogen)`

`v <- c(1, 2, 3, 4)`

`v[v > 2]`

`# matrix (2D homogen)`

`mat <- matrix(1:9, nrow = 3)`

`mat[2, 3]`

`t(mat)`

`# array (>=3D homogen)`

`arr3d <- array(1:12, dim = c(2, 3, 2))`

`arr3d[1, 2, 2]`

`# data frame (2D heterogen)`

`df <- data.frame(Name=c("A","B"), Age=c(20,21))`

`str(df)`

`# list (heterogen fleksibel)`

`lst <- list(vec=v, mat=mat, note="hello")`

`lst$note`

In [ ]:
# list (heterogen)
lst = [1, "A", True]
lst[0]

# dict (key-value)
dct = {"Name": "A", "Age": 20}
dct["Age"]

# numpy array (homogen, numerik)
import numpy as np
mat = np.arange(1, 10).reshape(3, 3)
mat[1, 2]
mat.T

# pandas DataFrame (heterogen tabular)
import pandas as pd
df = pd.DataFrame({"Name": ["A","B"], "Age": [20,21]})
df.dtypes

,0
Name,object
Age,int64


##6) Tugas di Kelas (mini-project 20–30 menit)



1. Bangun data panel “harga saham” sederhana dengan unit ii ∈ {A, B, C}, rentang 30 hari (date), dan variabel numerik price.

2. Pastikan data berada pada format long dengan struktur kolom minimal: (id, date, price). Jika data masih wide, lakukan konversi wide → long.

3. Hitung log-return per id (dihitung terpisah untuk setiap id).

4. Bangun dua fitur untuk klasifikasi dini dari setiap seri (per id), misalnya:
   - rolling mean 5-hari pada price, dan
   - rolling volatility 5-hari (simpangan baku) pada log-return.

5. Jelaskan secara eksplisit:
   - Observation (unit analisis) pada dataset panel ini,
   - Variable yang digunakan, dan
   - Skala pengukuran untuk tiap variabel (nominal/ordinal/interval/ratio).